In [2]:
# -------------------------------------------------------------------------------
# セル 1: インポート
# -------------------------------------------------------------------------------
import os
import tqdm
import time
import random
from copy import deepcopy
import abc
import json # JSONモジュールをインポート

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

import pennylane as qml
from pennylane import numpy as pnp # PennyLaneのnumpy
from pennylane.optimize import NesterovMomentumOptimizer
# from qiskit_aer import AerSimulator # PennyLane-Qiskitを使う場合

In [3]:
# -------------------------------------------------------------------------------
# ガイスターゲームの定数
# -------------------------------------------------------------------------------
BOARD_SIZE = 6
NUM_GHOSTS_PER_PLAYER = 8
GOOD_GHOST = "G"
BAD_GHOST = "B"
EMPTY = "."
PLAYER_A_ID = "A"
PLAYER_B_ID = "B"

def get_piece_str(player_id, ghost_kind):
    return player_id + ghost_kind

PLAYER_A_GOOD = get_piece_str(PLAYER_A_ID, GOOD_GHOST)
PLAYER_A_BAD = get_piece_str(PLAYER_A_ID, BAD_GHOST)
PLAYER_B_GOOD = get_piece_str(PLAYER_B_ID, GOOD_GHOST)
PLAYER_B_BAD = get_piece_str(PLAYER_B_ID, BAD_GHOST)

PLAYER_A_EXITS = [(0, 0), (0, BOARD_SIZE - 1)]
PLAYER_B_EXITS = [(BOARD_SIZE - 1, 0), (BOARD_SIZE - 1, BOARD_SIZE - 1)]

In [4]:
# -------------------------------------------------------------------------------
# PennyLaneデバイス設定 (グローバルに定義)
# -------------------------------------------------------------------------------
qubits_for_qnn_global = 4 # QNNで使用する量子ビット数。ガイスターの状態特徴量に応じて調整。
# dev_qnn_global = qml.device("default.qubit", wires=qubits_for_qnn_global)
# print(f"Global PennyLane device for QNN: {dev_qnn_global.name}")
# QNNを使用しない場合は上記はコメントアウトしてもよいが、CQCAgentの初期化でデバイスが必要になる。
# ダミーデバイスを渡すか、
# Agentを使わない場合はこのセクション自体不要。
# 今回はCQCAgentの定義を残すため、ダミーのdevを後でCNNAgentの学習前に定義します。
dev_qnn_global = qml.device("lightning.qubit", wires=qubits_for_qnn_global)

In [5]:
# -------------------------------------------------------------------------------
# セル : ガイスターゲームのルール (geister.py のクラス群をここに定義)
# -------------------------------------------------------------------------------
class Board:
    def __init__(self, size=BOARD_SIZE):
        self.size = size
        self.grid = [[EMPTY for _ in range(size)] for _ in range(size)]

    def get_piece(self, r, c):
        if 0 <= r < self.size and 0 <= c < self.size:
            return self.grid[r][c]
        return None

    def set_piece(self, r, c, piece_str):
        if 0 <= r < self.size and 0 <= c < self.size:
            self.grid[r][c] = piece_str
        # else: # 学習中はエラー出力を抑制することが多い
            # print(f"エラー: 盤外({r},{c})に駒を置こうとしました。")

    def remove_piece(self, r, c):
        if 0 <= r < self.size and 0 <= c < self.size:
            removed_piece = self.grid[r][c]
            self.grid[r][c] = EMPTY
            return removed_piece
        return None

    def display(self, current_player_id_for_display=None, reveal_opponent_pieces=False):
        header = "  " + " ".join(map(str, range(self.size)))
        print(header)
        for r_idx in range(self.size):
            row_display_list = [str(r_idx) + " "]
            for c_idx in range(self.size):
                piece = self.grid[r_idx][c_idx]
                if piece == EMPTY:
                    row_display_list.append(EMPTY)
                else:
                    player_of_piece = piece[0]
                    kind_of_piece = piece[1]
                    if (reveal_opponent_pieces or
                            current_player_id_for_display is None or
                            player_of_piece == current_player_id_for_display):
                        row_display_list.append(kind_of_piece)
                    else:
                        row_display_list.append("X")
            print(" ".join(row_display_list))
        print("-" * (self.size * 2 + 3)) # 表示調整

    def get_all_piece_positions(self, player_id=None):
        positions = []
        for r in range(self.size):
            for c in range(self.size):
                piece = self.grid[r][c]
                if piece != EMPTY:
                    if player_id is None or piece.startswith(player_id):
                        positions.append(((r, c), piece))
        return positions

class PieceSetupStrategy(abc.ABC):
    @abc.abstractmethod
    def setup_pieces(self, board: Board, player_id: str, pieces_to_place: list):
        pass

class RandomSetupStrategy(PieceSetupStrategy):
    def _get_valid_setup_positions_for_player(self, player_id: str, board_size: int) -> list:
        positions = []
        if player_id == PLAYER_A_ID: # Player A (盤面下側から見て手前2列の中央4マス)
            # row board_size - 2 (下から2行目)
            for c in range(1, board_size - 1): positions.append((board_size - 2, c))
            # row board_size - 1 (一番下の行)
            for c in range(1, board_size - 1): positions.append((board_size - 1, c))
        elif player_id == PLAYER_B_ID: # Player B (盤面上側から見て手前2列の中央4マス)
            # row 1 (上から2行目)
            for c in range(1, board_size - 1): positions.append((1, c))
            # row 0 (一番上の行)
            for c in range(1, board_size - 1): positions.append((0, c))
        return positions

    def setup_pieces(self, board: Board, player_id: str, pieces_to_place: list):
        if len(pieces_to_place) != NUM_GHOSTS_PER_PLAYER:
            raise ValueError(f"駒の数({len(pieces_to_place)})が期待値({NUM_GHOSTS_PER_PLAYER})と異なります。")
        
        valid_initial_positions = self._get_valid_setup_positions_for_player(player_id, board.size)
        if len(valid_initial_positions) < NUM_GHOSTS_PER_PLAYER:
             raise ValueError(f"配置可能マス({len(valid_initial_positions)})が駒の数({NUM_GHOSTS_PER_PLAYER})より少ないです。")

        shuffled_pieces = random.sample(pieces_to_place, len(pieces_to_place))
        # 配置可能なマスの中から、実際に配置する駒の数だけランダムに選ぶ
        shuffled_positions = random.sample(valid_initial_positions, NUM_GHOSTS_PER_PLAYER)

        for i in range(NUM_GHOSTS_PER_PLAYER):
            r, c = shuffled_positions[i]
            board.set_piece(r, c, shuffled_pieces[i])

class GeisterGame:
    def __init__(self, board_size=BOARD_SIZE,
                 setup_strategy_a: PieceSetupStrategy = None,
                 setup_strategy_b: PieceSetupStrategy = None):
        self.board_size = board_size
        self.setup_strategy_player_a = setup_strategy_a if setup_strategy_a else RandomSetupStrategy()
        self.setup_strategy_player_b = setup_strategy_b if setup_strategy_b else RandomSetupStrategy()
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.reset_board()

    def reset_board(self):
        self.board = Board(self.board_size)
        self.current_player = PLAYER_A_ID
        self.player_stats = {
            PLAYER_A_ID: {"captured_good": 0, "captured_bad": 0, "escaped_good": 0, "pieces_left": NUM_GHOSTS_PER_PLAYER},
            PLAYER_B_ID: {"captured_good": 0, "captured_bad": 0, "escaped_good": 0, "pieces_left": NUM_GHOSTS_PER_PLAYER},
        }
        self.game_over = False
        self.winner = None # None, PLAYER_A_ID, PLAYER_B_ID, "Draw"
        self._initialize_game_pieces()

    def _initialize_game_pieces(self):
        player_a_pieces = [PLAYER_A_GOOD] * 4 + [PLAYER_A_BAD] * 4
        self.setup_strategy_player_a.setup_pieces(self.board, PLAYER_A_ID, player_a_pieces)
        player_b_pieces = [PLAYER_B_GOOD] * 4 + [PLAYER_B_BAD] * 4
        self.setup_strategy_player_b.setup_pieces(self.board, PLAYER_B_ID, player_b_pieces)

    def get_player_of_piece(self, piece_str):
        if piece_str != EMPTY and len(piece_str) > 0: return piece_str[0]
        return None

    def get_kind_of_piece(self, piece_str):
        if piece_str != EMPTY and len(piece_str) > 1: return piece_str[1]
        return None

    def _is_valid_target(self, r, c, moving_player_id):
        if not (0 <= r < self.board.size and 0 <= c < self.board.size): return False
        target_piece = self.board.get_piece(r, c)
        if target_piece != EMPTY and self.get_player_of_piece(target_piece) == moving_player_id: return False
        return True

    def get_possible_actions(self, player_id=None):
        if player_id is None: player_id = self.current_player
        moves = []
        for (r, c), piece_str in self.board.get_all_piece_positions(player_id):
            for dr, dc in [(0, 1), (0, -1), (1, 0), (-1, 0)]:
                nr, nc = r + dr, c + dc
                if self._is_valid_target(nr, nc, player_id):
                    moves.append(((r, c), (nr, nc)))
        return moves
    
    def place(self, action, player_id): # For compatibility with OX game's Env
        if player_id != self.current_player: return False
        return self.make_move(action[0], action[1])

    def make_move(self, from_pos, to_pos):
        from_r, from_c = from_pos
        to_r, to_c = to_pos
        moving_piece_str = self.board.get_piece(from_r, from_c)

        if moving_piece_str == EMPTY or self.get_player_of_piece(moving_piece_str) != self.current_player: return False
        if not self._is_valid_target(to_r, to_c, self.current_player): return False

        opponent_id = PLAYER_B_ID if self.current_player == PLAYER_A_ID else PLAYER_A_ID
        
        captured_piece_str = self.board.get_piece(to_r, to_c)
        if captured_piece_str != EMPTY and self.get_player_of_piece(captured_piece_str) == opponent_id:
            captured_kind = self.get_kind_of_piece(captured_piece_str)
            if captured_kind == GOOD_GHOST: self.player_stats[self.current_player]["captured_good"] += 1
            elif captured_kind == BAD_GHOST: self.player_stats[self.current_player]["captured_bad"] += 1
            self.player_stats[opponent_id]["pieces_left"] -=1
        
        self.board.set_piece(to_r, to_c, moving_piece_str)
        self.board.remove_piece(from_r, from_c)

        moving_piece_kind = self.get_kind_of_piece(moving_piece_str)
        if moving_piece_kind == GOOD_GHOST:
            exits = PLAYER_A_EXITS if self.current_player == PLAYER_A_ID else PLAYER_B_EXITS
            if (to_r, to_c) in exits:
                self.player_stats[self.current_player]["escaped_good"] += 1
                self.board.remove_piece(to_r, to_c)
                self.player_stats[self.current_player]["pieces_left"] -= 1
        
        self.check_win_condition()
        if not self.game_over:
            self.switch_player()
        return True
        
    def switch_player(self):
        self.current_player = PLAYER_B_ID if self.current_player == PLAYER_A_ID else PLAYER_A_ID

    def check_win_condition(self):
        stats_a = self.player_stats[PLAYER_A_ID]
        stats_b = self.player_stats[PLAYER_B_ID]
        if stats_a["captured_good"] >= 4: self.game_over = True; self.winner = PLAYER_A_ID
        elif stats_b["captured_good"] >= 4: self.game_over = True; self.winner = PLAYER_B_ID
        elif stats_a["captured_bad"] >= 4: self.game_over = True; self.winner = PLAYER_B_ID # B wins if A's bad are all captured
        elif stats_b["captured_bad"] >= 4: self.game_over = True; self.winner = PLAYER_A_ID # A wins if B's bad are all captured
        elif stats_a["escaped_good"] >= 1: self.game_over = True; self.winner = PLAYER_A_ID
        elif stats_b["escaped_good"] >= 1: self.game_over = True; self.winner = PLAYER_B_ID

    def checkwinner_for_reward(self, player_id):
        """
        Returns +1 if player_id wins, -1 if loses, 0 for draw or ongoing.
        """
        if not self.game_over:
            return 0.0
        if self.winner == player_id:
            return 1.0
        elif self.winner == "Draw":
            return 0.0
        else:
            return -1.0
        
    def gameover(self): # For OX game's Env compatibility
        return self.game_over

    def checkwinner(self): # For OX game's Env reward calculation
        if self.game_over: return self.winner
        return None

    def get_state(self, player_id_perspective: str) -> torch.tensor:
        # チャンネル0: player_id_perspective の良駒
        # チャンネル1: player_id_perspective の悪駒
        # チャンネル2: 相手の駒 (種類不明)
        # チャンネル3: player_id_perspective の駒が置かれているマス(良悪問わず)
        # チャンネル4: player_id_perspective の脱出口
        # チャンネル5: 相手の脱出口
        # (オプション) チャンネル6: 盤全体で空いているマス
        num_channels = 6 #チャンネル数を増やす
        state = torch.zeros((num_channels, self.board_size, self.board_size), dtype=torch.float32, device=self.device)
        
        opponent_id = PLAYER_B_ID if player_id_perspective == PLAYER_A_ID else PLAYER_A_ID
        my_exits = PLAYER_A_EXITS if player_id_perspective == PLAYER_A_ID else PLAYER_B_EXITS
        opponent_exits = PLAYER_B_EXITS if player_id_perspective == PLAYER_A_ID else PLAYER_A_EXITS

        for r in range(self.board_size):
            for c in range(self.board_size):
                piece = self.board.get_piece(r,c)
                if piece != EMPTY:
                    owner = self.get_player_of_piece(piece)
                    kind = self.get_kind_of_piece(piece)
                    if owner == player_id_perspective:
                        state[3,r,c] = 1.0 # 自分の駒がある
                        if kind == GOOD_GHOST: state[0, r, c] = 1.0
                        elif kind == BAD_GHOST: state[1, r, c] = 1.0
                    elif owner == opponent_id:
                        state[2, r, c] = 1.0 # 相手の駒
                # else: # 空きマス
                #     state[6,r,c] = 1.0 # オプション：空きマスチャネル
                
                if (r,c) in my_exits: state[4,r,c] = 1.0
                if (r,c) in opponent_exits: state[5,r,c] = 1.0
        return state

    def display_board(self, sleep_seconds=0, reveal_opponent_pieces=False): # Renamed for clarity
        self.board.display(current_player_id_for_display=self.current_player, reveal_opponent_pieces=reveal_opponent_pieces)
        # print(f"Current player: {self.current_player}")
        # for p_id in [PLAYER_A_ID, PLAYER_B_ID]:
        #     stats = self.player_stats[p_id]
        #     print(f"Player {p_id}: Captured(G:{stats['captured_good']},B:{stats['captured_bad']}), Escaped(G:{stats['escaped_good']}), Pieces Left({stats['pieces_left']})")
        if sleep_seconds > 0: time.sleep(sleep_seconds)

In [ ]:
# --- QNNComponent (元のコードから、一部修正の可能性あり) ---
class QNNComponent:
    def __init__(self, n_qubits: int):
        self.n_qubits = n_qubits

    def TPE(self, theta, reps: int = 1): # Tensor Product Encoder
        for _ in range(reps):
            for qubit_index in range(self.n_qubits):
                # Ensure theta has enough elements
                if qubit_index < len(theta):
                    qml.RX(theta[qubit_index], wires=qubit_index)

    def HEE(self, theta, reps: int = 1): # Hardware Efficient Ansatz / Encoder
        # Ensure theta has enough elements for all rotations
        param_idx = 0
        for _ in range(reps):
            for qubit_index in range(self.n_qubits):
                if param_idx < len(theta):
                    qml.RY(theta[param_idx], wires=qubit_index) # Use RY for HEE generally
                    param_idx +=1
                else: break # Not enough params
            if param_idx >= len(theta) and self.n_qubits > 0 : break # Stop if out of params
            
            for qubit_index in range(self.n_qubits - 1):
                qml.CZ(wires=[qubit_index, qubit_index + 1]) # Use CZ for HEE
            if self.n_qubits > 1: # Circular entanglement for the last qubit
                 qml.CZ(wires=[self.n_qubits-1, 0])


    def ZFeatureMap(self, theta, reps: int = 1):
        for _ in range(reps):
            for i in range(self.n_qubits):
                if i < len(theta):
                    qml.Hadamard(wires=i)
                    qml.RZ(2.0 * theta[i], wires=i)

    def ZZFeatureMap(self, theta, reps: int = 1):
        pi_val = pnp.pi
        for _ in range(reps):
            for i in range(self.n_qubits):
                if i < len(theta):
                    qml.Hadamard(wires=i)
                    qml.RZ(2.0 * theta[i], wires=i)

            for i in range(self.n_qubits - 1):
                if i + 1 < len(theta):
                    qml.CNOT(wires=[i, i + 1])
                    angle = 2.0 * (pi_val - theta[i]) * (pi_val - theta[i + 1])
                    qml.RZ(angle, wires=i + 1)
                    qml.CNOT(wires=[i, i + 1])

    def RealAmplitudes(self, theta, reps: int = 1):
        idx = 0
        for i in range(self.n_qubits):
            if idx < len(theta):
                qml.RY(theta[idx], wires=i)
                idx += 1

        for _ in range(reps):
            for i in range(self.n_qubits - 1):
                qml.CNOT(wires=[i, i + 1])

            for i in range(self.n_qubits):
                if idx < len(theta):
                    qml.RZ(theta[idx], wires=i)
                    idx += 1
            for i in range(self.n_qubits):
                if idx < len(theta):
                    qml.RY(theta[idx], wires=i)
                    idx += 1
            if idx >= len(theta): break


    def EfficientSU2(self, theta, reps: int = 1):
        # theta should have (reps+1) * 2 * n_qubits parameters
        idx = 0
        for i in range(self.n_qubits): # Initial layer
            if idx + 1 < len(theta):
                qml.RY(theta[idx], wires=i); idx += 1
                qml.RZ(theta[idx], wires=i); idx += 1
            else: break
        if idx >= len(theta) and reps > 0: return # Not enough params for even one rep

        for _ in range(reps):
            # Entanglement: all-to-all CZs (or CNOTs) or a specific pattern
            # Simplified to a chain for now, similar to HEE / RealAmplitudes
            for i in range(self.n_qubits -1 ): qml.CZ(wires=[i,i+1])
            if self.n_qubits > 1: qml.CZ(wires=[self.n_qubits-1,0]) # Circular

            for i in range(self.n_qubits): # Subsequent layers
                if idx + 1 < len(theta):
                    qml.RY(theta[idx], wires=i); idx += 1
                    qml.RZ(theta[idx], wires=i); idx += 1
                else: break
            if idx >= len(theta): break


    def make_circuit(self, embedding_type, ansatz_type, input_params, weight_params, exp_or_prob, feature_map_reps=1, ansatz_reps=1):
        # Feature map
        if embedding_type in ["ZFeatureMap", "ZZFeatureMap", "TPE", "HEE"]:
            if input_params.shape[0] != self.n_qubits:
                qml.AngleEmbedding(input_params, wires=range(self.n_qubits), rotation='X')
            else:
                getattr(self, embedding_type)(input_params, feature_map_reps)
        else:
            qml.AngleEmbedding(input_params, wires=range(self.n_qubits), rotation='X')

        # Ansatz
        if hasattr(self, ansatz_type):
            getattr(self, ansatz_type)(weight_params, ansatz_reps)
        else:
            raise ValueError(f"Unknown ansatz type: {ansatz_type}")

        # Measurement
        if exp_or_prob == "exp":
            return [qml.expval(qml.PauliZ(wires=i)) for i in range(self.n_qubits)]
        elif exp_or_prob == "prob":
            return qml.probs(wires=range(self.n_qubits))
        else:
            raise ValueError(f"Unknown measurement type: {exp_or_prob}")

In [ ]:
# --- CNN Model (CCNN2_Geister) ---
class CCNN2_Geister(nn.Module):
    def __init__(self, input_channels=6, board_size=BOARD_SIZE, num_outputs=BOARD_SIZE*BOARD_SIZE):
        super().__init__()
        self.board_size = board_size
        self.input_channels = input_channels
        
        self.conv1 = nn.Conv2d(self.input_channels, 32, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        # Optional: Max Pooling
        # self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2) # 6x6 -> 3x3 if applied after conv2
        
        # Calculate flattened size dynamically
        with torch.no_grad():
            dummy_input = torch.zeros(1, self.input_channels, self.board_size, self.board_size)
            # x = self.pool1(self.relu2(self.conv2(self.relu1(self.conv1(dummy_input))))) # If pooling
            x = self.relu2(self.conv2(self.relu1(self.conv1(dummy_input)))) # No pooling
            self.flatten_size = x.numel() // x.shape[0] # numel gives total elements, divide by batch size

        self.flatten = nn.Flatten()
        self.linear = nn.Linear(self.flatten_size, num_outputs) # No Tanh, raw Q-values

    def forward(self, state):
        x = self.relu1(self.conv1(state))
        x = self.relu2(self.conv2(x))
        # x = self.pool1(x) # If pooling
        x = self.flatten(x)
        x = self.linear(x)
        # Reshape to (batch_size, board_height, board_width) for Q-value map
        x = x.view(-1, self.board_size, self.board_size)
        return x

class CNN_QNN_CNN_Geister(nn.Module):
    def __init__(self, dev, embedding_type: str, ansatz_type: str, 
                 n_qubits_qnn=4, exp_or_prob="exp", 
                 feature_map_reps=1, ansatz_reps=1,
                 input_channels_cnn=6, board_size_cnn=6, 
                 cnn_fc_out_features=4,  # CNNからQNNに送る特徴数 = QNN入力数
                 qnn_fc_out_features=36):  # Q値出力 = 6x6マス
        super().__init__()
        self.dev = dev
        self.embedding_type = embedding_type
        self.ansatz_type = ansatz_type
        self.exp_or_prob = exp_or_prob
        self.feature_map_reps = feature_map_reps
        self.ansatz_reps = ansatz_reps
        self.n_qubits = n_qubits_qnn
        self.board_size_cnn = board_size_cnn

        # CNN部分
        self.cnn_feature_extractor = nn.Sequential(
            nn.Conv2d(input_channels_cnn, 16, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(),
            nn.Flatten()
        )
        with torch.no_grad():
            dummy_input = torch.zeros(1, input_channels_cnn, board_size_cnn, board_size_cnn)
            cnn_flat_dim = self.cnn_feature_extractor(dummy_input).shape[1]
        self.fc_to_qnn = nn.Linear(cnn_flat_dim, cnn_fc_out_features)

        # QNN部分の重み
        if ansatz_type == "RealAmplitudes":
            num_weights = (ansatz_reps + 1) * self.n_qubits
        elif ansatz_type == "EfficientSU2":
            num_weights = (ansatz_reps + 1) * 2 * self.n_qubits
        else:
            raise ValueError(f"Unsupported ansatz type: {ansatz_type}")
        self.q_weights = nn.Parameter(torch.randn(num_weights, requires_grad=True))

        # QNNコンポーネント
        self.qcomp = QNNComponent(n_qubits=self.n_qubits)

        @qml.qnode(self.dev, interface="torch", diff_method="adjoint")
        def qnode(inputs, weights):
            return self.qcomp.make_circuit(
                embedding_type=self.embedding_type,
                ansatz_type=self.ansatz_type,
                input_params=inputs,
                weight_params=weights,
                exp_or_prob=self.exp_or_prob,
                feature_map_reps=self.feature_map_reps,
                ansatz_reps=self.ansatz_reps
            )

        self.qnode = qnode  # メンバに保存
        self.qnn_output_dim = self.n_qubits if self.exp_or_prob == "exp" else 2**self.n_qubits
        self.fc_from_qnn = nn.Linear(self.qnn_output_dim, qnn_fc_out_features)

    def forward(self, state):
        cnn_features = self.cnn_feature_extractor(state)  # shape: (B, N)
        qnn_input_features = self.fc_to_qnn(cnn_features)  # shape: (B, n_qubits)

        qnn_outputs = []
        for i in range(qnn_input_features.shape[0]):
            input_vector = qnn_input_features[i]
            qnn_output = self.qnode(input_vector, self.q_weights)
            # ↓ここを追加
            if isinstance(qnn_output, list):
                qnn_output = torch.tensor(qnn_output,dtype=torch.float32, device=self.fc_from_qnn.weight.device)
            qnn_outputs.append(qnn_output)

        qnn_outputs_tensor = torch.stack(qnn_outputs)  # shape: (B, qnn_output_dim)
        x = self.fc_from_qnn(qnn_outputs_tensor)
        return x.view(-1, self.board_size_cnn, self.board_size_cnn)

In [ ]:
# --- エージェントクラス (Agent, Human, RandomPolicy, CNNAgent_Geister, CQCAgent_Geister) ---
class Agent(abc.ABC):
    def __init__(self, player_id: str, game: GeisterGame):
        self.player_id = player_id
        self.game = game
        self.eval_mode = True # Start in eval mode

    def train_mode_on(self): # Changed name
        self.eval_mode = False

    def eval_mode_on(self): # Changed name
        self.eval_mode = True

    def check_state(self) -> torch.tensor:
        return self.game.get_state(self.player_id)

    def check_actions(self) -> list:
        return self.game.get_possible_actions(self.player_id)
    
    @abc.abstractmethod
    def action(self, possible_moves: list) -> tuple: # Returns ((fr,fc),(tr,tc)) or None
        pass

class Human(Agent):
    def __init__(self, player_id: str, game: GeisterGame):
        super().__init__(player_id, game)
        self.eval_mode_on()

    def action(self, possible_moves: list) -> tuple:
        if not possible_moves: return None
        print(f"プレイヤー {self.player_id} の番です。選択可能な手:")
        for i, ((fr, fc), (tr, tc)) in enumerate(possible_moves):
            piece = self.game.board.get_piece(fr,fc)
            kind = self.game.get_kind_of_piece(piece) if piece != EMPTY else '?'
            print(f"{i}: ({fr},{fc}) の {kind}オバケ を ({tr},{tc}) へ")
        while True:
            try:
                choice_str = input(f"移動する駒の番号を選んでください (0-{len(possible_moves)-1}): ")
                if not choice_str: continue # 空入力を無視
                choice = int(choice_str)
                if 0 <= choice < len(possible_moves):
                    return possible_moves[choice]
                else:
                    print(f"無効な番号です。0から{len(possible_moves)-1}の間で入力してください。")
            except ValueError:
                print("数値を入力してください。")
            except Exception as e:
                print(f"予期せぬエラー: {e}")


class RandomPolicy(Agent):
    def __init__(self, player_id: str, game: GeisterGame):
        super().__init__(player_id, game)
        self.eval_mode_on()
        self.episode_count = 0  # ← これを追加

    def update(self, *args, **kwargs):
        return None  # 何もしない

    def action(self, possible_moves: list) -> tuple:
        if not possible_moves: return None
        return random.choice(possible_moves)

class CNNAgent_Geister(Agent):
    def __init__(self, player_id:str, game:GeisterGame, network_name="CCNN2_Geister", 
                 board_size=BOARD_SIZE, input_channels=6, elo=1500, epsilon=0.1, lr=0.001): # input_channelsを6に変更
        super().__init__(player_id, game)
        self.discount = 0.99
        self.epsilon_start = epsilon
        self.epsilon_end = 0.01
        self.epsilon_decay = 5000 # エピソード数で減衰
        self.epsilon = epsilon
        self.lr = lr
        self.alpha = 0.0 # L2正則化なしに一旦変更
        self.board_size = board_size
        self.input_channels = input_channels # 状態表現のチャネル数
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.elo = elo
        self.episode_count = 0 # epsilon減衰用

        if network_name == "CCNN2_Geister":
            self.NN = CCNN2_Geister(input_channels=self.input_channels, board_size=self.board_size, num_outputs=self.board_size*self.board_size).to(self.device)
        else:
            raise ValueError(f"Unknown network name: {network_name}")
        
        self.optimizer = optim.Adam(self.NN.parameters(), lr=self.lr)
        self.eval_mode_on()

    def get_qvalues_map(self, state_tensor: torch.tensor) -> torch.tensor:
        state_tensor = state_tensor.unsqueeze(0).to(self.device)
        qvalues_map = self.NN(state_tensor)
        return qvalues_map.squeeze(0)

    def _get_the_best_action_and_qvalue(self, state_tensor: torch.tensor, possible_moves: list):
        if not possible_moves:
            return None, -float('inf')
        with torch.no_grad():
            qvalues_map = self.get_qvalues_map(state_tensor)
        best_move = None
        best_q_value = -float('inf')
        # 複数の手が同じ最大Q値を持つ場合、ランダムに選ぶ
        best_moves_candidates = []

        for move in possible_moves:
            _from_pos, to_pos = move
            to_r, to_c = to_pos
            q_value = qvalues_map[to_r, to_c].item()
            
            if q_value > best_q_value:
                best_q_value = q_value
                best_moves_candidates = [move]
            elif q_value == best_q_value:
                best_moves_candidates.append(move)
        
        if best_moves_candidates:
            best_move = random.choice(best_moves_candidates)
        elif possible_moves : # 全てのQ値が-infの場合など（通常はありえない）
             best_move = random.choice(possible_moves)
             _from_pos, to_pos = best_move # best_q_valueは-infのまま
             best_q_value = qvalues_map[to_pos[0], to_pos[1]].item()


        return best_move, best_q_value

    def action(self, possible_moves: list) -> tuple:
        current_state_tensor = self.check_state()
        
        # epsilon decay
        if not self.eval_mode:
            self.epsilon = self.epsilon_end + (self.epsilon_start - self.epsilon_end) * \
                           np.exp(-1. * self.episode_count / self.epsilon_decay)

        if not self.eval_mode and random.random() < self.epsilon:
            if not possible_moves: return None
            return random.choice(possible_moves)
        else:
            best_action, _ = self._get_the_best_action_and_qvalue(current_state_tensor, possible_moves)
            return best_action

    def update(self, state_tensor: torch.tensor, action: tuple, reward: float, next_state_tensor: torch.tensor, done: bool):
        if self.eval_mode: return None
        if action is None: return None # 行動がない場合は更新スキップ

        _from_pos, to_pos = action
        to_r, to_c = to_pos
        
        q_value_map_current = self.get_qvalues_map(state_tensor)
        old_qvalue = q_value_map_current[to_r, to_c]

        target_q_value = torch.tensor(reward, dtype=torch.float32, device=self.device)
        
        if not done:
            with torch.no_grad():
                # 次の状態 s' における次の手番プレイヤーの最適な行動のQ値 (max_a' Q(s',a'))
                # 相手の手番だとしても、自分のNNを使って相手の最善手を推定し、その負の値を使う
                next_player_id = self.game.current_player # Make_move後なので、相手の手番になっている
                next_possible_moves = self.game.get_possible_actions(next_player_id)
                
                if not next_possible_moves: # 相手が動けない -> 自分の勝ちが確定したようなもの
                    max_next_q_for_opponent = 0.0 # 相手はこれ以上Q値を改善できない
                else:
                    # 相手も同じ戦略(このNN)を使うと仮定。相手視点の状態を取得。
                    next_state_tensor_opponent_view = self.game.get_state(next_player_id)
                    _ , max_next_q_for_opponent = self._get_the_best_action_and_qvalue(next_state_tensor_opponent_view, next_possible_moves)
                    if max_next_q_for_opponent == -float('inf'): max_next_q_for_opponent = 0.0

                # 相手の最大Q値の符号を反転させたものが、自分にとっての次の状態の価値
                target_q_value += self.discount * (-max_next_q_for_opponent)
        
        loss = nn.functional.huber_loss(old_qvalue, target_q_value.detach())
        if self.alpha > 0:
            l2 = torch.tensor(0., requires_grad=True, device=self.device)
            for w in self.NN.parameters(): l2 = l2 + torch.norm(w)**2
            loss = loss + self.alpha * l2

        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        return loss.item()

class CQCAgent_Geister(Agent):
    def __init__(self, player_id:str, game:GeisterGame, dev_qnn,
                 embedding_type: str, ansatz_type: str, n_qubits_qnn=4, 
                 exp_or_prob="exp", feature_map_reps=1, ansatz_reps=1,
                 input_channels_cnn=6, board_size_cnn=BOARD_SIZE, 
                 cnn_fc_out_features=4, # QNNへの入力特徴数 (n_qubits_qnnと一致させる想定)
                 elo=1500, epsilon=0.1, lr=0.001):
        super().__init__(player_id, game)
        self.discount = 0.99
        self.epsilon_start = epsilon
        self.epsilon_end = 0.01
        self.epsilon_decay = 5000
        self.epsilon = epsilon
        self.lr = lr
        self.board_size = board_size_cnn
        self.input_channels = input_channels_cnn
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.dev_qnn = dev_qnn
        self.elo = elo
        self.episode_count = 0

        self.HNN = CNN_QNN_CNN_Geister(
            dev=self.dev_qnn, embedding_type=embedding_type, ansatz_type=ansatz_type,
            n_qubits_qnn=n_qubits_qnn, exp_or_prob=exp_or_prob,
            feature_map_reps=feature_map_reps, ansatz_reps=ansatz_reps,
            input_channels_cnn=self.input_channels, board_size_cnn=self.board_size,
            cnn_fc_out_features=cnn_fc_out_features,
        ).to(self.device)
        
        self.optimizer = optim.Adam(self.HNN.parameters(), lr=self.lr)
        self.eval_mode_on()

    def get_qvalues_map(self, state_tensor: torch.tensor) -> torch.tensor:
        state_tensor = state_tensor.unsqueeze(0).to(self.device)
        qvalues_map = self.HNN(state_tensor)
        return qvalues_map.squeeze(0)

    def _get_the_best_action_and_qvalue(self, state_tensor: torch.tensor, possible_moves: list):
        # (CNNAgent_Geister と同じロジック)
        if not possible_moves: return None, -float('inf')
        with torch.no_grad(): qvalues_map = self.get_qvalues_map(state_tensor)
        best_move = None; best_q_value = -float('inf'); best_moves_candidates = []
        for move in possible_moves:
            _from_pos, to_pos = move; to_r, to_c = to_pos
            q_value = qvalues_map[to_r, to_c].item()
            if q_value > best_q_value: best_q_value = q_value; best_moves_candidates = [move]
            elif q_value == best_q_value: best_moves_candidates.append(move)
        if best_moves_candidates: best_move = random.choice(best_moves_candidates)
        elif possible_moves:
             best_move = random.choice(possible_moves)
             _from_pos, to_pos = best_move
             best_q_value = qvalues_map[to_pos[0], to_pos[1]].item()
        return best_move, best_q_value
        
    def action(self, possible_moves: list) -> tuple:
        # (CNNAgent_Geister と同じロジック)
        current_state_tensor = self.check_state()
        if not self.eval_mode:
            self.epsilon = self.epsilon_end + (self.epsilon_start - self.epsilon_end) * \
                           np.exp(-1. * self.episode_count / self.epsilon_decay)
        if not self.eval_mode and random.random() < self.epsilon:
            if not possible_moves: return None
            return random.choice(possible_moves)
        else:
            best_action, _ = self._get_the_best_action_and_qvalue(current_state_tensor, possible_moves)
            return best_action

    def update(self, state_tensor: torch.tensor, action: tuple, reward: float, next_state_tensor: torch.tensor, done: bool):
        # (CNNAgent_Geister と同じロジック)
        if self.eval_mode: return None
        if action is None: return None
        _from_pos, to_pos = action; to_r, to_c = to_pos
        q_value_map_current = self.get_qvalues_map(state_tensor)
        old_qvalue = q_value_map_current[to_r, to_c]
        target_q_value = torch.tensor(reward, dtype=torch.float32, device=self.device)
        if not done:
            with torch.no_grad():
                next_player_id = self.game.current_player 
                next_possible_moves = self.game.get_possible_actions(next_player_id)
                if not next_possible_moves: max_next_q_for_opponent = 0.0
                else:
                    next_state_tensor_opponent_view = self.game.get_state(next_player_id)
                    _ , max_next_q_for_opponent = self._get_the_best_action_and_qvalue(next_state_tensor_opponent_view, next_possible_moves)
                    if max_next_q_for_opponent == -float('inf'): max_next_q_for_opponent = 0.0
                target_q_value += self.discount * (-max_next_q_for_opponent) # Minimax Q-like update
        loss = nn.functional.huber_loss(old_qvalue, target_q_value.detach())
        self.optimizer.zero_grad(); loss.backward(); self.optimizer.step()
        return loss.item()
    
    

In [ ]:
# --- 対戦環境クラス (Env_Geister) ---
class Env_Geister:
    def __init__(self, agent1: Agent, agent2: Agent, game: GeisterGame):
        self.game = game
        self.agent1 = agent1
        self.agent2 = agent2
        self.agent1.player_id = PLAYER_A_ID
        self.agent1.game = self.game
        self.agent2.player_id = PLAYER_B_ID
        self.agent2.game = self.game
        self.max_turns_per_game = 200
        self.loss_history_agent1 = []
        self.loss_history_agent2 = []

    def play_one_game(self, visualize=False, train_agents=True):
        self.game.reset_board()
        if train_agents:
            self.agent1.train_mode_on(); self.agent2.train_mode_on()
            self.agent1.episode_count +=1; self.agent2.episode_count +=1 # For epsilon decay
        else:
            self.agent1.eval_mode_on(); self.agent2.eval_mode_on()

        # (s, a, r, s_next, done) storage for each agent for a full episode or a step
        # For simplicity, we update after each agent's turn if it's not the first turn
        # (state_tensor, action_tuple)
        last_info_agent1 = {"state": None, "action": None, "reward_pending":0.0, "next_state_pending":None, "done_pending":False}
        last_info_agent2 = {"state": None, "action": None, "reward_pending":0.0, "next_state_pending":None, "done_pending":False}

        for turn_count in range(self.max_turns_per_game):
            if visualize: self.game.display_board(reveal_opponent_pieces=True) # Reveal for debugging/vis

            current_player_id = self.game.current_player
            active_agent = self.agent1 if current_player_id == PLAYER_A_ID else self.agent2
            opponent_agent = self.agent2 if current_player_id == PLAYER_A_ID else self.agent1
            
            current_state_tensor = active_agent.check_state()
            possible_moves = active_agent.check_actions()

            if not possible_moves: # No legal moves
                self.game.winner = opponent_agent.player_id
                self.game.game_over = True
                # Update the previous player (opponent) who made the winning move
                if train_agents:
                    if opponent_agent.player_id == PLAYER_A_ID and last_info_agent1["state"] is not None:
                        loss = self.agent1.update(last_info_agent1["state"], last_info_agent1["action"], 1.0, current_state_tensor, True)
                        if loss is not None: self.loss_history_agent1.append(loss)
                    elif opponent_agent.player_id == PLAYER_B_ID and last_info_agent2["state"] is not None:
                        loss = self.agent2.update(last_info_agent2["state"], last_info_agent2["action"], 1.0, current_state_tensor, True)
                        if loss is not None: self.loss_history_agent2.append(loss)
                break

            action = active_agent.action(possible_moves)
            if action is None: # Agent failed to choose an action
                if possible_moves: action = random.choice(possible_moves)
                else: # Should be caught above
                    self.game.winner = opponent_agent.player_id
                    self.game.game_over = True
                    break
            
            # Execute the action and get the outcome
            # The (s,a) is for 'active_agent'. 'r' and 's_next' are observed after this.
            # The 'opponent_agent' was the one who acted previously to lead to 'current_state_tensor' for 'active_agent'.
            # So, we update 'opponent_agent' first.
            
            # If opponent had a pending state/action, update opponent
            if train_agents:
                info_to_update = last_info_agent1 if active_agent == self.agent2 else last_info_agent2
                if info_to_update["state"] is not None:
                    # Reward for opponent is 0 if game not over, or -1 if active_agent (self) won on this turn
                    # This s' for opponent is current_state_tensor for active_agent
                    reward_for_opponent = 0.0 # Intermediate reward
                    done_for_opponent = False
                    # No, this update logic is complex. Let's simplify for now:
                    # Update happens for the agent *after* their action and *after* opponent's response (or game end).
                    pass


            # Store current s,a for active_agent
            if active_agent == self.agent1:
                last_info_agent1 = {"state": current_state_tensor.clone(), "action": action}
            else:
                last_info_agent2 = {"state": current_state_tensor.clone(), "action": action}

            self.game.make_move(action[0], action[1]) # Game state changes, player possibly switches
            
            # Now, the game state is s' (s_next) resulting from active_agent's action.
            # The reward r is for active_agent.
            # The game might have ended.
            reward_active = self.game.checkwinner_for_reward(active_agent.player_id)
            done_active = self.game.gameover()
            next_state_tensor_for_active = active_agent.check_state() # s' from active_agent's perspective
            
            if train_agents:
                if active_agent == self.agent1:
                    loss = self.agent1.update(last_info_agent1["state"], last_info_agent1["action"], reward_active, next_state_tensor_for_active, done_active)
                    if loss is not None: self.loss_history_agent1.append(loss)
                else: # active_agent == self.agent2
                    loss = self.agent2.update(last_info_agent2["state"], last_info_agent2["action"], reward_active, next_state_tensor_for_active, done_active)
                    if loss is not None: self.loss_history_agent2.append(loss)
            
            if done_active:
                break
        
        if not self.game.game_over and turn_count >= self.max_turns_per_game -1 :
            self.game.winner = "Draw" # Max turns reached
            self.game.game_over = True
            # Final update for both if pending
            if train_agents:
                reward_draw = 0.0
                final_state_A = self.game.get_state(PLAYER_A_ID)
                final_state_B = self.game.get_state(PLAYER_B_ID)
                if last_info_agent1["state"] is not None:
                    loss = self.agent1.update(last_info_agent1["state"], last_info_agent1["action"], reward_draw, final_state_A, True)
                    if loss is not None: self.loss_history_agent1.append(loss)
                if last_info_agent2["state"] is not None:
                    loss = self.agent2.update(last_info_agent2["state"], last_info_agent2["action"], reward_draw, final_state_B, True)
                    if loss is not None: self.loss_history_agent2.append(loss)
        
        return self.game.winner

    def start_training(self, episodes, visualize_interval=0, train_agents=True, model_save_interval=100, model_dir_prefix="./models_geister"):
        wins_A = 0; wins_B = 0; draws = 0
        for i in range(episodes):
            winner = self.play_one_game(visualize=(visualize_interval > 0 and (i + 1) % visualize_interval == 0), train_agents=train_agents)
            if winner == PLAYER_A_ID: wins_A += 1
            elif winner == PLAYER_B_ID: wins_B += 1
            else: draws += 1

            if (i + 1) % 100 == 0:
                total_played = wins_A + wins_B + draws
                print(f"--- Episode {i + 1}/{episodes} ---")
                print(f"Agent A Wins: {wins_A} ({wins_A/total_played:.2%}), Agent B Wins: {wins_B} ({wins_B/total_played:.2%}), Draws: {draws} ({draws/total_played:.2%})")
                avg_loss_A = np.mean(self.loss_history_agent1[-500:]) if self.loss_history_agent1 else float('nan')
                avg_loss_B = np.mean(self.loss_history_agent2[-500:]) if self.loss_history_agent2 else float('nan')
                print(f"Avg Loss A (last 500): {avg_loss_A:.4f}, Avg Loss B (last 500): {avg_loss_B:.4f}")
                print(f"Agent A Epsilon: {self.agent1.epsilon if hasattr(self.agent1, 'epsilon') else 'N/A'}")
                print(f"Agent B Epsilon: {self.agent2.epsilon if hasattr(self.agent2, 'epsilon') else 'N/A'}")

            if train_agents and model_save_interval > 0 and (i + 1) % model_save_interval == 0:
                # ----- Agent A -----
                model_dir_A = f"{model_dir_prefix}_agentA/"
                if not os.path.exists(model_dir_A): os.makedirs(model_dir_A)

                if hasattr(self.agent1, 'NN') and self.agent1.NN is not None:
                    model_path = os.path.join(model_dir_A, f"agentA_eps{i+1}.pth")
                    torch.save(self.agent1.NN.state_dict(), model_path)
                    print(f"Agent A model saved to {model_path}")
                if hasattr(self.agent1, 'QNN') and self.agent1.QNN is not None:
                    qmodel_path = os.path.join(model_dir_A, f"agentA_qnn_eps{i+1}.pth")
                    torch.save(self.agent1.QNN.state_dict(), qmodel_path)
                    print(f"Agent A QNN saved to {qmodel_path}")

                # ----- Agent B -----
                if self.agent2 != self.agent1:
                    model_dir_B = f"{model_dir_prefix}_agentB/"
                    if not os.path.exists(model_dir_B): os.makedirs(model_dir_B)

                    if hasattr(self.agent2, 'NN') and self.agent2.NN is not None:
                        model_path = os.path.join(model_dir_B, f"agentB_eps{i+1}.pth")
                        torch.save(self.agent2.NN.state_dict(), model_path)
                        print(f"Agent B model saved to {model_path}")
                    if hasattr(self.agent2, 'QNN') and self.agent2.QNN is not None:
                        qmodel_path = os.path.join(model_dir_B, f"agentB_qnn_eps{i+1}.pth")
                        torch.save(self.agent2.QNN.state_dict(), qmodel_path)
                        print(f"Agent B QNN saved to {qmodel_path}")

        print("Training finished.")
        final_total = wins_A + wins_B + draws
        if final_total > 0:
            print(f"Final Score: A Wins: {wins_A} ({wins_A/final_total:.2%}), B Wins: {wins_B} ({wins_B/final_total:.2%}), Draws: {draws} ({draws/final_total:.2%})")

In [ ]:
# --- 学習・評価関数の定義 (簡略化) ---
def run_geister_cnn_training(episodes=1000):
    print("--- Training CNN Agent for Geister ---")
    game_instance = GeisterGame()
    # 状態表現のチャネル数 (GeisterGame.get_state の出力に合わせる)
    # 例: 自良(1),自悪(1),敵駒(1),自駒位置(1),自脱出口(1),敵脱出口(1) -> 6チャネル
    INPUT_CHANNELS_FOR_GEISTER = 6 

    agent_a = CNNAgent_Geister(PLAYER_A_ID, game_instance, input_channels=INPUT_CHANNELS_FOR_GEISTER, epsilon=0.5, lr=0.0005)
    agent_b = CNNAgent_Geister(PLAYER_B_ID, game_instance, input_channels=INPUT_CHANNELS_FOR_GEISTER, epsilon=0.5, lr=0.0005) # 自己対戦
    # agent_b = RandomPolicy(PLAYER_B_ID, game_instance) # 対ランダム

    env = Env_Geister(agent_a, agent_b, game_instance)
    env.start_training(episodes, visualize_interval=0, train_agents=True, model_save_interval=episodes // 5) # 5回保存
    
    print("\n--- Evaluating Trained CNN Agent A vs Random ---")
    agent_a.eval_mode_on() # 評価モード
    eval_env_A_vs_Random = Env_Geister(agent_a, RandomPolicy(PLAYER_B_ID, game_instance), game_instance)
    eval_env_A_vs_Random.start_training(episodes=100, visualize_interval=0, train_agents=False)

    print("\n--- Evaluating Trained CNN Agent B vs Random ---")
    agent_b.eval_mode_on() # 評価モード
    eval_env_B_vs_Random = Env_Geister(RandomPolicy(PLAYER_A_ID, game_instance), agent_b, game_instance)
    eval_env_B_vs_Random.start_training(episodes=100, visualize_interval=0, train_agents=False)


# --- CQCAgent の学習・評価関数 (注意：dev_qnn_global の初期化が必要) ---
def run_geister_cqcnn_training(episodes=100, n_qbits=4, cnn_out_feat=4):
    print(f"--- Training CQCNN Agent (Qubits: {n_qbits}, CNN->QNN Feat: {cnn_out_feat}) ---")
    global dev_qnn_global # グローバルデバイスを使う
    if dev_qnn_global is None or len(dev_qnn_global.wires) != n_qbits:
        dev_qnn_global = qml.device("default.qubit", wires=n_qbits)
        print(f"Initialized QNN device: {dev_qnn_global.name} with {n_qbits} qubits.")

    game_instance_q = GeisterGame()
    INPUT_CHANNELS_FOR_GEISTER = 6

    # cnn_fc_out_features は QNNComponent のエンコーディング層が期待する入力次元数
    # AngleEmbedding を使う場合は cnn_fc_out_features と n_qubits_qnn が一致しなくても良いが、
    # ZFeatureMapなどを使う場合は一致させる必要がある (またはスライス/パディング)
    # ここでは cnn_fc_out_features = n_qubits_qnn と仮定。
    
    agent_a_q = CQCAgent_Geister(
        PLAYER_A_ID, game_instance_q, dev_qnn_global,
        embedding_type="AngleEmbedding", ansatz_type="RealAmplitudes",
        n_qubits_qnn=n_qbits,
        input_channels_cnn=INPUT_CHANNELS_FOR_GEISTER,
        board_size_cnn=BOARD_SIZE,
        cnn_fc_out_features=n_qbits, # QNNへの入力特徴数 (n_qubitsと一致させる)
        epsilon=0.5, lr=0.0005
    )
    agent_b_q = RandomPolicy(PLAYER_B_ID, game_instance_q) # 対ランダム

    env_q = Env_Geister(agent_a_q, agent_b_q, game_instance_q)
    env_q.start_training(episodes, visualize_interval=0, train_agents=True, model_save_interval=episodes // 2)

    print("\n--- Evaluating Trained CQCNN Agent A vs Random ---")
    agent_a_q.eval_mode_on()
    agent_a_q.epsilon = 0.0 # 評価時はランダム性なし
    eval_env_Q_vs_Random = Env_Geister(agent_a_q, RandomPolicy(PLAYER_B_ID, game_instance_q), game_instance_q)
    eval_env_Q_vs_Random.start_training(episodes=100, visualize_interval=0, train_agents=False)

In [ ]:
# -------------------------------------------------------------------------------
# メイン実行ブロック
# -------------------------------------------------------------------------------
if __name__ == '__main__': # Jupyter Notebookではこのブロックは直接実行されないが、.py化を考慮
    # 1. CNN Agentの学習と評価
    run_geister_cnn_training(episodes=3000) # エピソード数を調整
    # 2. CQCNN Agentの学習と評価 (必要であればコメントを外して実行)
    # 注意: QNNの学習は非常に時間がかかる可能性があります。
    #qubits_for_qnn_run = 4  # QNNで使う量子ビット数
    #cnn_to_qnn_features_run = 4 # CNNからの特徴量をQNNの入力にする数 (上記qubitsと合わせるか、AngleEmbedding等で調整)
    #run_geister_cqcnn_training(episodes=3000, n_qbits=qubits_for_qnn_run, cnn_out_feat = cnn_to_qnn_features_run)


--- Training CQCNN Agent (Qubits: 4, CNN->QNN Feat: 4) ---
--- Episode 100/3000 ---
Agent A Wins: 70 (70.00%), Agent B Wins: 25 (25.00%), Draws: 5 (5.00%)
Avg Loss A (last 500): 0.0143, Avg Loss B (last 500): nan
Agent A Epsilon: 0.4902973499203101
Agent B Epsilon: N/A
--- Episode 200/3000 ---
Agent A Wins: 120 (60.00%), Agent B Wins: 54 (27.00%), Draws: 26 (13.00%)
Avg Loss A (last 500): 0.0049, Avg Loss B (last 500): nan
Agent A Epsilon: 0.48078682518463833
Agent B Epsilon: N/A
--- Episode 300/3000 ---
Agent A Wins: 169 (56.33%), Agent B Wins: 81 (27.00%), Draws: 50 (16.67%)
Avg Loss A (last 500): 0.0071, Avg Loss B (last 500): nan
Agent A Epsilon: 0.4714646214562819
Agent B Epsilon: N/A
--- Episode 400/3000 ---
Agent A Wins: 211 (52.75%), Agent B Wins: 106 (26.50%), Draws: 83 (20.75%)
Avg Loss A (last 500): 0.0085, Avg Loss B (last 500): nan
Agent A Epsilon: 0.4623270097294515
Agent B Epsilon: N/A
--- Episode 500/3000 ---
Agent A Wins: 269 (53.80%), Agent B Wins: 131 (26.20%), Draws

In [ ]:
import torch
import os

# 保存先ディレクトリ
save_dir = "./models_geister_final/agentA/"
os.makedirs(save_dir, exist_ok=True)

# --- 保存処理 ---
if hasattr(agent_a_q, 'NN') and agent_a_q.NN is not None:
    cnn_path = os.path.join(save_dir, "agentA_CNN_final.pth")
    torch.save(agent_a_q.NN.state_dict(), cnn_path)
    print(f"✅ CNN part of Agent A saved to: {cnn_path}")

if hasattr(agent_a_q, 'QNN') and agent_a_q.QNN is not None:
    qnn_path = os.path.join(save_dir, "agentA_QNN_final.pth")
    torch.save(agent_a_q.QNN.state_dict(), qnn_path)
    print(f"✅ QNN part of Agent A saved to: {qnn_path}")

# --- 最終モデルの保存 ---
save_dir = "./models_geister_final/agentA/"
os.makedirs(save_dir, exist_ok=True)

if hasattr(agent_a_q, 'NN') and agent_a_q.NN is not None:
    torch.save(agent_a_q.NN.state_dict(), os.path.join(save_dir, "agentA_CNN_final.pth"))
    print("✅ CNN model saved")

if hasattr(agent_a_q, 'QNN') and agent_a_q.QNN is not None:
    torch.save(agent_a_q.QNN.state_dict(), os.path.join(save_dir, "agentA_QNN_final.pth"))
    print("✅ QNN model saved")

NameError: name 'agent_a_q' is not defined

In [ ]:

    # 3. 人間 vs 学習済みAI の対戦 (例)
    # print("\n--- Human vs Trained CNN Agent ---")
    # game_vs_human = GeisterGame()
    # trained_cnn_path = "./models_geister_cnn_agentA/agentA_eps10000.pth" # 保存したモデルのパス
    # if os.path.exists(trained_cnn_path):
    #     human_player = Human(PLAYER_A_ID, game_vs_human)
    #     ai_opponent = CNNAgent_Geister(PLAYER_B_ID, game_vs_human, input_channels=6)
    #     ai_opponent.NN.load_state_dict(torch.load(trained_cnn_path, map_location=ai_opponent.device))
    #     ai_opponent.eval_mode_on()
        
    #     env_human_vs_ai = Env_Geister(human_player, ai_opponent, game_vs_human)
    #     env_human_vs_ai.play_one_game(visualize=True, train_agents=False)
    # else:
    #     print(f"Trained model not found at {trained_cnn_path}. Skipping Human vs AI game.")